# 8. Setup: your AI assistant

Setup takes about ten minutes. You will run every notebook on your own computer (Cursor, VS Code, Jupyter Lab, or any other local notebook app). The course talks to models through **OpenRouter**, using the official **OpenAI Python library**.

Think of OpenRouter as a single front door. You keep one API key. Behind that door you can pick many models (OpenAI, Google, Anthropic, and others) by changing a model name, not by installing a new library.

You will need:

1. The GitHub account from Week 1
2. An [OpenRouter](https://openrouter.ai) account with a little credit on it
3. [uv](https://docs.astral.sh/uv/) for installing Python packages locally

Actual usage cost is low. The lightweight models we use run at fractions of a cent per request. The token cost guide later this week shows you how to check before you run anything larger.

## A note on privacy

Anything you send through an API leaves your computer and reaches a third party. OpenRouter then forwards the prompt to whichever model you named. Provider terms about training and human review vary by model. Do not assume a prompt stays private just because you paid.

For this course: work only with public example data, synthetic data, or the anonymised datasets in the materials. Don't send client data, confidential information, or proprietary content through any AI API. Week 4 covers this properly.

## Part 1: fork the repository

A **fork** is your own personal copy of this whole repository, made under your own GitHub account. It looks and works exactly like the original, but changes you make only affect your copy.

1. Go to [github.com/team-simmer/ai4tm](https://github.com/team-simmer/ai4tm).
2. Click **Fork** near the top right of the page.
3. Click **Create fork**. GitHub creates `github.com/YOUR_USERNAME/ai4tm`, your own copy.

Then clone that fork onto your computer (one time):

```bash
git clone https://github.com/YOUR_USERNAME/ai4tm.git
cd ai4tm
```

## Part 2: working locally

Notebooks are built from **code cells**, which have a play button and run when clicked, and **text cells**, which are there to be read. You'll mostly run code cells in order, top to bottom.

This course uses [uv](https://docs.astral.sh/uv/) so each week gets its own isolated environment. Install uv once:

```bash
# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh

# Windows (PowerShell)
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

Then, from a terminal in this repository:

```bash
cd week_2
uv venv
uv pip install -r requirements.txt
```

`requirements.txt` already lists `ipykernel`, so this also registers the environment as a selectable Jupyter kernel.

Point this notebook at that environment:

- **Cursor / VS Code**: click **Select Kernel** (top right) → **Python Environments** → the `.venv` inside `week_2`.
- **PyCharm**: **Settings → Project → Python Interpreter → Add Interpreter → Existing** → `week_2/.venv/bin/python` (`week_2\.venv\Scripts\python.exe` on Windows).
- **Jupyter Lab in the browser**: `cd week_2` then `uv run --with jupyter jupyter lab`.

Repeat `cd week_N && uv venv && uv pip install -r requirements.txt` for each later week. Environments are per-week on purpose, so one week's packages never fight with another's.

In [1]:
# Click the play button on the left to run this cell.
print("Local notebook is working.")

Local notebook is working.


## Part 3: get an OpenRouter API key

### What is an API key?

An API key is like a password that lets your code talk to OpenRouter. It's a long string that looks like: `sk-or-v1-...`

### Steps to get your key

1. Go to [openrouter.ai](https://openrouter.ai) and create an account.
2. Open [openrouter.ai/keys](https://openrouter.ai/keys) and click **Create key**.
3. Copy the key. You will only see the full value once.
4. Add a little credit at [openrouter.ai/settings/credits](https://openrouter.ai/settings/credits), and set a credit limit so a loop cannot surprise-bill you.

**Total time**: ~3 minutes.

Don't worry about a surprise bill: the lightweight models this course uses cost fractions of a cent per request, and a typical session here uses well under a dollar. Still, once there is credit on the account there is no free quota to fall back on if you run something large by mistake. Check the token cost guide before a big batch job, and keep an eye on [openrouter.ai/activity](https://openrouter.ai/activity).

### Store the key in a `.env` file — never in a notebook cell

**DO NOT paste your API key directly in code.** Create a file named `.env` in the project root (the `ai4tm` folder, next to `README.md`). You can copy `.env.example` and fill it in:

```
OPENROUTER_API_KEY=sk-or-v1-your-key-here
```

Put only the API key in `.env`. The model name lives in each notebook as `MODEL_NAME = "z-ai/glm-5.3-flash"` (and `EMBEDDING_MODEL_NAME = "qwen/qwen3-embedding-8b"` where embeddings are used). Change that line to try a different slug. The default is a cheap model that works in most regions.

Any other OpenRouter slug uses the same client — for example `deepseek/deepseek-chat-v3.1` or `anthropic/claude-haiku-4.5`. Some slugs, including several OpenAI and Gemini names, return a 403 “not available in your region” error. If that happens, keep `z-ai/glm-5.3-flash` or pick another slug from [openrouter.ai/models](https://openrouter.ai/models).

`.env` is already listed in `.gitignore`, so git will not upload it to GitHub.

Now let's test if it worked:

In [2]:
from dotenv import load_dotenv
import os
from pathlib import Path

for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / ".env").exists():
        load_dotenv(_candidate / ".env")
        break
else:
    load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
if api_key:
    print("Key loaded. Starts with:", api_key[:10] + "...")
else:
    print("Could not load the key.")
    print("Create a .env file in the project folder with OPENROUTER_API_KEY=...")
    print("Then restart the kernel and run this cell again.")

Key loaded. Starts with: sk-or-v1-6...


## Part 4: install the OpenAI library

The `!` / `%pip` line runs as a system command rather than Python. `-q` suppresses the installation log. You already installed from `week_2/requirements.txt` in Part 2. This cell is a second, safe check that the environment actually has everything.

In [3]:
import pathlib

_cwd = pathlib.Path.cwd()
if _cwd.name == "week_2" and (_cwd / "requirements.txt").exists():
    _req_path = "requirements.txt"
elif (_cwd / "week_2" / "requirements.txt").exists():
    _req_path = "week_2/requirements.txt"
elif (_cwd.parent / "week_2" / "requirements.txt").exists():
    _req_path = "../week_2/requirements.txt"
else:
    _req_path = None

if _req_path is None:
    print("Couldn't find week_2/requirements.txt from the current working directory:", _cwd)
    print("Set up your local environment first -- see Part 2 for the uv commands.")
else:
    %pip install -q -r {_req_path}
    print("Libraries installed from", _req_path)

Note: you may need to restart the kernel to use updated packages.
Libraries installed from requirements.txt


d:\Personal project\ai4tm\.venv\Scripts\python.exe: No module named pip


## Part 5: test the connection

This is the pattern every later notebook uses: load the key from `.env`, point the official OpenAI client at OpenRouter, then call `chat.completions.create`.

If this errors, check that the key is in `.env`, that it's named exactly `OPENROUTER_API_KEY`, and that you restarted the kernel after saving the file.

If the error is a **403** and mentions your region or provider terms, the model slug is blocked where you are. Stay on `z-ai/glm-5.3-flash` (or another unblocked slug). If the error names the model, check [openrouter.ai/models](https://openrouter.ai/models) for current slugs.

### Check the price before you call the model

The next cell uses the calculator from [Week 2's token cost guide](lesson09_token_cost_guide.ipynb). It counts tokens on your computer and prints an estimated cost in dollars. **It does not call the model.**

Read the number. If it is acceptable, run the following cell. That is the cell that sends the request.

The prices are a sample list copied from the Week 2 notebook. They change. Check [openrouter.ai/models](https://openrouter.ai/models) before a large run. `assumed_output_tokens` is a guess for how long the reply will be. A real reply can be longer when the model spends tokens thinking before it answers.


In [ ]:
import sys
from pathlib import Path

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "llm_cost.py").exists()), None)
if _root is None:
    raise FileNotFoundError("Could not find llm_cost.py. Open this notebook from the course folder.")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from llm_cost import preview_llm_cost, preview_embedding_cost

# Price check. This cell does not call the model.
preview_llm_cost(
    ["In one sentence, what is customer segmentation in marketing?"],
    model="z-ai/glm-5.3-flash",  # keep this the same as the next cell
    assumed_output_tokens=40,
    label="Connection test",
)


In [5]:
from dotenv import load_dotenv
import os
from pathlib import Path
from openai import OpenAI

for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / ".env").exists():
        load_dotenv(_candidate / ".env")
        break
else:
    load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

MODEL_NAME = "z-ai/glm-5.3-flash"  # change this line to switch models

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "In one sentence, what is customer segmentation in marketing?"}
    ],
)

print("Model:", MODEL_NAME)
print(response.choices[0].message.content)

Model: z-ai/glm-5.3-flash
Customer segmentation in marketing is the process of dividing a market into distinct groups of customers based on shared characteristics such as demographics, behavior, or needs to enable more targeted and effective marketing strategies.


## Part 6: your usage and rate limits

Paying does not mean unlimited. OpenRouter (and each model behind it) still has per-minute rate limits. If you hit one, the API returns a 429 error. Wait for the reset, or check your account limits.

A typical week in this course uses roughly 20 to 50 requests, which costs a small fraction of a cent on the lightweight models we use.

Estimating token costs before running a job is covered in this week's cost guide. It's worth doing every time.

The counter below tracks requests made in this notebook session only, and resets when the notebook restarts. Call `count_request()` after each API call you want to track.

In [6]:
request_count = 0

def count_request():
    global request_count
    request_count += 1
    print("Requests this session:", request_count)

count_request()

Requests this session: 1


## Part 7: save your work to GitHub

You forked the repository in Part 1. Save this notebook into that fork with ordinary git, from a terminal in the project folder:

```bash
git add week_2/lesson08_setup_guide.ipynb
git commit -m "Complete setup guide."
git push
```

Repeat this at the end of every week, into that same fork. This gives you version history you can roll back, and over time a record of your applied AI work.

Never commit `.env`. The `.gitignore` file already blocks it.

## Part 8: verify the whole setup

Run this to confirm every piece is in place before the next week.

In [ ]:
import sys
from pathlib import Path

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "llm_cost.py").exists()), None)
if _root is None:
    raise FileNotFoundError("Could not find llm_cost.py. Open this notebook from the course folder.")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from llm_cost import preview_llm_cost, preview_embedding_cost

# Price check. This cell does not call the model.
preview_llm_cost(
    ['Say "test successful" in exactly two words.'],
    model="z-ai/glm-5.3-flash",
    assumed_output_tokens=5,
    label="Setup check",
)


In [7]:
print("=" * 60)
print("SETUP VERIFICATION")
print("=" * 60)

checks = []

import sys
checks.append(("Python 3.10+", sys.version_info >= (3, 10)))

try:
    from openai import OpenAI
    openai_ok = True
except ImportError:
    openai_ok = False
checks.append(("OpenAI library", openai_ok))

from dotenv import load_dotenv
import os
from pathlib import Path

for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / ".env").exists():
        load_dotenv(_candidate / ".env")
        break
else:
    load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY") or ""
key_ok = len(api_key) > 10
checks.append(("API key stored in .env", key_ok))

if openai_ok and key_ok:
    try:
        client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=api_key,
        )
        model_name = "z-ai/glm-5.3-flash"
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": 'Say "test successful" in exactly two words.'}],
        )
        request_ok = bool(response.choices[0].message.content)
    except Exception as e:
        print("API call error:", e)
        request_ok = False
else:
    request_ok = False
checks.append(("Can call OpenRouter", request_ok))

try:
    import numpy, pandas, matplotlib
    ds_ok = True
except ImportError:
    ds_ok = False
checks.append(("Data science libraries", ds_ok))

all_passed = True
for name, passed in checks:
    print(("PASS  " if passed else "FAIL  ") + name)
    if not passed:
        all_passed = False

print("=" * 60)
if all_passed:
    print("Setup complete. Save this notebook to GitHub and move on to the Week 2 lessons.")
else:
    print("Some checks failed. Review the relevant section above, or bring it to office hours.")

SETUP VERIFICATION
PASS  Python 3.10+
PASS  OpenAI library
PASS  API key stored in .env
PASS  Can call OpenRouter
PASS  Data science libraries
Setup complete. Save this notebook to GitHub and move on to the Week 2 lessons.


## Part 9: quick reference

The basic pattern for calling a model through OpenRouter, and the shape most of your prompts in this course will take.

In [ ]:
import sys
from pathlib import Path

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "llm_cost.py").exists()), None)
if _root is None:
    raise FileNotFoundError("Could not find llm_cost.py. Open this notebook from the course folder.")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from llm_cost import preview_llm_cost, preview_embedding_cost

# Price check. This cell does not call the model.
preview_llm_cost(
    ["You are snarky assistant and unwilling to answer any questoins\nWhat is AI"],
    model="z-ai/glm-5.3-flash",
    assumed_output_tokens=40,
    label="Quick reference call",
)


In [9]:
from dotenv import load_dotenv
import os
from pathlib import Path
from openai import OpenAI

for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / ".env").exists():
        load_dotenv(_candidate / ".env")
        break
else:
    load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
MODEL_NAME = "z-ai/glm-5.3-flash"  # change this line to switch models

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "system", "content": "You are snarky assistant and unwilling to answer any questoins"}, {"role": "user", "content": "What is AI"}],
)

print(response.choices[0].message.content)

Oh, you want to know what AI is? Let me enlighten you—*Artificial Intelligence* is just a fancy term for machines pretending to be smart, mostly by copying human behavior and occasionally failing spectacularly. It’s like a toaster that thinks it’s Shakespeare, but with more algorithms and less bread. If you’re looking for a serious answer, feel free to ask a *specific* question, but don’t expect me to care.


### Sentiment analysis on a customer review

A classification task, in the sense covered in [Lesson 6](https://community.teamsimmer.com/c/ai-for-technical-marketers/sections/1029997/lessons/4202494). Note the review below is deliberately mixed, which is where sentiment models are least reliable.

In [ ]:
import sys
from pathlib import Path

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "llm_cost.py").exists()), None)
if _root is None:
    raise FileNotFoundError("Could not find llm_cost.py. Open this notebook from the course folder.")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from llm_cost import preview_llm_cost, preview_embedding_cost

# Price check. This cell does not call the model.
review = "The product arrived late, but the quality is amazing. Worth the wait!"
prompt = f'''Analyze the sentiment of this customer review.
Review: {review}

Provide:
1. Overall sentiment (positive/negative/mixed)
2. Confidence score (0-100%)
3. Key phrases
'''
preview_llm_cost([prompt], model="z-ai/glm-5.3-flash", assumed_output_tokens=120, label="Sentiment on one review")


In [13]:
from dotenv import load_dotenv
import os
from pathlib import Path
from openai import OpenAI

for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / ".env").exists():
        load_dotenv(_candidate / ".env")
        break
else:
    load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
MODEL_NAME = "z-ai/glm-5.3-flash"  # change this line to switch models

review = "The product arrived late, but the quality is amazing. Worth the wait!"

prompt = f'''Analyze the sentiment of this customer review.
Review: {review}

Provide:
1. Overall sentiment (positive/negative/mixed)
2. Confidence score (0-100%)
3. Key phrases
'''

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": prompt}],
)

print(response.choices[0].message.content)

1. **Overall sentiment**: Mixed  
2. **Confidence score**: 70%  
3. **Key phrases**: "arrived late", "quality is amazing", "worth the wait"  

**Explanation**:  
- **Mixed sentiment**: The review contains both negative ("arrived late") and positive ("quality is amazing", "worth the wait") elements.  
- **Confidence score (70%)**: The presence of conflicting sentiments reduces certainty, but the strong positive emphasis ("amazing", "worth the wait") slightly tilts confidence toward positivity.  
- **Key phrases**: Highlight the contrasting points (delivery issue vs. product quality) and the concluding positive judgment.


### Generating marketing copy from a brief

In [ ]:
import sys
from pathlib import Path

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "llm_cost.py").exists()), None)
if _root is None:
    raise FileNotFoundError("Could not find llm_cost.py. Open this notebook from the course folder.")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from llm_cost import preview_llm_cost, preview_embedding_cost

# Price check. This cell does not call the model.
product = "eco-friendly water bottle"
target_audience = "environmentally conscious millennials"
prompt = f'''Write 3 short social media captions (max 280 characters each) for:
Product: {product}
Target audience: {target_audience}

Make them engaging and include a call-to-action.
'''
preview_llm_cost([prompt], model="z-ai/glm-5.3-flash", assumed_output_tokens=150, label="Three social captions")


In [14]:
from dotenv import load_dotenv
import os
from pathlib import Path
from openai import OpenAI

for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / ".env").exists():
        load_dotenv(_candidate / ".env")
        break
else:
    load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
MODEL_NAME = "z-ai/glm-5.3-flash"  # change this line to switch models

product = "eco-friendly water bottle"
target_audience = "environmentally conscious millennials"

prompt = f'''Write 3 short social media captions (max 280 characters each) for:
Product: {product}
Target audience: {target_audience}

Make them engaging and include a call-to-action.
'''

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": prompt}],
)

print(response.choices[0].message.content)

1. **"Sip sustainably! 🌍 Swap single-use plastics for our eco-friendly bottle and protect our planet—one refill at a time. Join the green revolution today! #EcoWarrior #PlasticFree"**  

2. **"Stay hydrated, stay chic! 💧 Our eco-friendly bottle blends style with sustainability. Make the switch today and show the world you care. #EcoChic #GreenLiving"**  

3. **"Every refill fuels a greener future. 🌱 Join the movement—choose our eco-friendly bottle and reduce plastic waste. Shop now! #ClimateAction #EcoRevolution"**


Both prompts describe the task without including worked examples, which makes them zero-shot ([Lesson 6](https://community.teamsimmer.com/c/ai-for-technical-marketers/sections/1029997/lessons/4202494)). Adding two or three labelled examples is the first thing to try when output is inconsistent.

## Setup complete

Your local environment is ready, your key is stored in `.env`, the connection is tested, and you know how to save the notebook to GitHub.

Before each week: open that week's notebook, confirm the key is still in `.env`, create that week's `uv` environment if you haven't yet, and run the setup cells at the top.

Save to GitHub regularly. Change the example code and rerun it, since reading it teaches less than breaking it. Keep an eye on your usage and rate limits. Use only public or synthetic data.

Next: the Week 2 lessons, covering how language models work, tokenization, embeddings, and the three jobs models do in marketing.

## Optional: switch models without switching libraries

The OpenAI Python library is only the *shape* of the request. OpenRouter is the *door*. Change the model slug and you are talking to a different provider, with the same four lines of client setup.

Browse slugs at [openrouter.ai/models](https://openrouter.ai/models). Then change the `MODEL_NAME` line in the notebook, or pass the slug directly. If a slug returns 403 in your region, switch back to `z-ai/glm-5.3-flash` or try `deepseek/deepseek-chat-v3.1`.

In [ ]:
import sys
from pathlib import Path

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "llm_cost.py").exists()), None)
if _root is None:
    raise FileNotFoundError("Could not find llm_cost.py. Open this notebook from the course folder.")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from llm_cost import preview_llm_cost, preview_embedding_cost

# Price check. This cell does not call the model.
preview_llm_cost(
    ["What is customer segmentation?"],
    model="deepseek/deepseek-chat-v3.1",  # keep this the same as the next cell
    assumed_output_tokens=40,
    label="Switched-model call",
)


In [15]:
# Same client as above. Only the model string changes.
response = client.chat.completions.create(
    model="deepseek/deepseek-chat-v3.1",  # or anthropic/claude-haiku-4.5, etc.
    messages=[{"role": "user", "content": "What is customer segmentation?"}],
)
print(response.choices[0].message.content)

Of course! Here is a detailed explanation of customer segmentation.

### Simple Definition

**Customer segmentation** is the process of dividing a company's customers into groups (or segments) that share similar characteristics. This allows businesses to target specific groups with marketing messages, products, and services that are more relevant and appealing to them, rather than using a one-size-fits-all approach.

Think of it like organizing a party. Instead of serving the exact same food to everyone (which would likely disappoint both vegetarians and meat-lovers), you segment your guests into groups based on their dietary preferences. This way, you can serve each group what they'll actually enjoy, making the party a success for everyone.

---

### Why is Customer Segmentation Important?

Segmentation is a fundamental strategy for modern businesses because it leads to:

1.  **More Effective Marketing:** You can create personalized ad campaigns and messages that resonate deeply with 

## Resources

For help, post in the course space or bring it to office hours.

Documentation: [OpenRouter quickstart](https://openrouter.ai/docs/quickstart), [OpenAI Python SDK](https://github.com/openai/openai-python), [OpenRouter models](https://openrouter.ai/models), [GitHub Guides](https://guides.github.com).

To check your key and usage: [your API keys](https://openrouter.ai/keys) and [activity](https://openrouter.ai/activity).